# Decoding Customer Value: A SQL-Driven Retention Strategy

## Project Objective

The objective of this project is to develop a customer retention strategy for a Direct-to-Consumer (D2C) fashion brand using customer purchase behaviour.

Since the dataset contains no timestamps, loyalty labels, or churn indicators, customer loyalty cannot be assumed. Instead, it will be constructed using observable behavioural variables and validated using purchase amount and internal consistency.

The project follows a four-stage analytics workflow:

1. Data Preparation & Feature Engineering (Python)
2. Customer Segmentation & Business Analysis (MySQL)
3. Executive Dashboard (Power BI)
4. Retention Strategy & Recommendations

This notebook focuses exclusively on preparing a clean analytical dataset and engineering meaningful business features that will later be used in SQL and Power BI.
    

# Phase 1 :- Data Preparation

Transforming the raw customer dataset into a clean and reliable analytical dataset.

In [60]:
import pandas as pd
import numpy as np

In [61]:
# Load the dataset

df = pd.read_csv("Dataset(1).csv")

# Display the first five rows

df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


### Initial Dataset Inspection

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

### Standardizing Column Names

In [63]:
#Converting all column names to snake_case.
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("(", "", regex=False)
      .str.replace(")", "", regex=False)
      .str.replace("/", "_")
)

# Display updated column names

df.columns.tolist()

['customer_id',
 'age',
 'gender',
 'item_purchased',
 'category',
 'purchase_amount_usd',
 'location',
 'size',
 'color',
 'season',
 'review_rating',
 'subscription_status',
 'shipping_type',
 'discount_applied',
 'promo_code_used',
 'previous_purchases',
 'payment_method',
 'frequency_of_purchases']

### Duplicate Record Check

In [64]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate Records Found: {duplicate_rows}")

if duplicate_rows > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicate records removed.")
else:
    print("No duplicate records found.")

Duplicate Records Found: 0
No duplicate records found.


### Missing Value Analysis

In [65]:
missing_values = df.isnull().sum()

missing_summary = (
    missing_values[missing_values > 0]
    .sort_values(ascending=False)
    .to_frame(name="Missing Values")
)

missing_summary

,Missing Values
review_rating,37


### Missing Value Treatment

In [66]:
#Review rating returned 37 missing values, assigning values based on median rating within each product category (instead of using the overall dataset median).
if df["review_rating"].isnull().sum() > 0:

    df["review_rating"] = (
        df.groupby("category")["review_rating"]
          .transform(lambda x: x.fillna(x.median()))
    )

    print("Review ratings assigned using category-wise median.")

else:
    print("No missing values detected in Review Rating.")

Review ratings assigned using category-wise median.


### Redundant Column Check

In [67]:
redundant_check = (
    df["discount_applied"] ==
    df["promo_code_used"]
).all()

print("Are both columns identical?", redundant_check)

Are both columns identical? True


In [68]:
if redundant_check:

    df.drop(columns=["promo_code_used"], inplace=True)

    print("'promo_code_used' removed.")

else:

    print("Both variables retained.")

'promo_code_used' removed.


### Data Type Verification

In [69]:
df.dtypes

customer_id                 int64
age                         int64
gender                     object
item_purchased             object
category                   object
purchase_amount_usd         int64
location                   object
size                       object
color                      object
season                     object
review_rating             float64
subscription_status        object
shipping_type              object
discount_applied           object
previous_purchases          int64
payment_method             object
frequency_of_purchases     object
dtype: object

# Phase 2 :- Feature Engineering
The engineered features will be:

1. Purchase Frequency Days
2. Satisfaction Flag
3. Value Tier
4. Customer Health Index (CHI)
5. Promotion Dependency Index (PDI)

Each feature has a clear business interpretation and will later be used in SQL-based customer segmentation and retention analysis.

### Feature 1 :- Purchase Frequency Days

In [70]:
# Inspect all purchase frequency categories

display(
    pd.DataFrame(
        sorted(df["frequency_of_purchases"].unique()),
        columns=["Frequency Category"]
    )
)

,Frequency Category
0,Annually
1,Bi-Weekly
2,Every 3 Months
3,Fortnightly
4,Monthly
5,Quarterly
6,Weekly


In [71]:
#Assigning numerical values to the frequency category
frequency_mapping = {
    "Weekly": 7,
    "Fortnightly": 15,
    "Bi-Weekly": 14,
    "Monthly": 30,
    "Quarterly": 90,
    "Every 3 Months": 91,
    "Annually": 365
}

df["purchase_frequency_days"] = (
    df["frequency_of_purchases"]
      .map(frequency_mapping)
)

In [72]:
display(
    df[
        [
            "frequency_of_purchases",
            "purchase_frequency_days"
        ]
    ].head(10)
)

,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,15
1,Fortnightly,15
2,Weekly,7
3,Weekly,7
4,Annually,365
5,Weekly,7
6,Quarterly,90
7,Weekly,7
8,Annually,365
9,Quarterly,90


## Feature 2 :- Satisfaction Flag

In [73]:
#Assigning 'Satisifed' or 'Needs Attention' based on their comparison with the median value
median_rating = df["review_rating"].median()

print(f"Median Review Rating : {median_rating}")

Median Review Rating : 3.8


In [74]:
df["satisfaction_flag"] = np.where(
    df["review_rating"] >= median_rating,
    "Satisfied",
    "Needs Attention"
)

In [75]:
#An almost equal split was found in the distribution of the satisfaction flag
print(df["satisfaction_flag"].value_counts())

satisfaction_flag
Satisfied          1955
Needs Attention    1945
Name: count, dtype: int64


### Feature 3 :- Value Tier

In [76]:
purchase_quartiles = df["purchase_amount_usd"].quantile(
    [0.25, 0.50, 0.75]
)

purchase_quartiles

0.25    39.0
0.50    60.0
0.75    81.0
Name: purchase_amount_usd, dtype: float64

In [77]:
df["value_tier"] = pd.qcut(
    df["purchase_amount_usd"],
    q=4,
    labels=[
        "Low Value",
        "Emerging",
        "High Value",
        "Premium"
    ]
)

In [78]:
display(
    df[
        [
            "purchase_amount_usd",
            "value_tier"
        ]
    ].head(10)
)

,purchase_amount_usd,value_tier
0,53,Emerging
1,64,High Value
2,73,High Value
3,90,Premium
4,49,Emerging
5,20,Low Value
6,85,Premium
7,34,Low Value
8,97,Premium
9,31,Low Value


## Feature 4 :- Customer Health Index (CHI)

Customer Health Index (CHI) is a composite business index, summarizing the overall **strength of a customer's relationship with the brand**, using the relation of the contributing variables with **Purchase Amount (USD)**. Three variables are considered:

- Previous Purchases
- Purchase Frequency (Days)
- Review Rating

### Step 4.1: Measuring Relationships with Purchase Amount

In [79]:
# Variables contributing to Customer Health Index

chi_variables = [
    "previous_purchases",
    "purchase_frequency_days",
    "review_rating"
]

# Calculate correlations(Pearsons) with Purchase Amount

correlations = (
    df[chi_variables + ["purchase_amount_usd"]]
    .corr()["purchase_amount_usd"]
    .drop("purchase_amount_usd")
)

display(correlations.to_frame(name="Correlation"))

,Correlation
previous_purchases,0.008063
purchase_frequency_days,0.009515
review_rating,0.029659


In [80]:
# Store original correlations

original_correlations = correlations.copy()

# Convert to absolute values for weight calculation

absolute_correlations = correlations.abs()

display(
    absolute_correlations.to_frame(
        name="Absolute Correlation"
    )
)

,Absolute Correlation
previous_purchases,0.008063
purchase_frequency_days,0.009515
review_rating,0.029659


### Step 4.2: Converting Correlations into Weights

In [81]:
chi_weights = (
    absolute_correlations /
    absolute_correlations.sum()
)

display(
    (chi_weights * 100)
    .round(2)
    .to_frame(name="Weight (%)")
)
#The distribution of weights shows that the review_rating is most important in determining the CHI, followed by the purchase_frequency_days

,Weight (%)
previous_purchases,17.07
purchase_frequency_days,20.14
review_rating,62.79


### Step 4.3: Normalizing the Variables

In [82]:
# Normalize variables

df["purchase_score"] = (
    df["previous_purchases"] /
    df["previous_purchases"].max()
)

df["rating_score"] = (
    df["review_rating"] /
    df["review_rating"].max()
)

df["frequency_score"] = (
    df["purchase_frequency_days"] /
    df["purchase_frequency_days"].max()
)

### Step 4.4: Orienting the Variables

In [83]:
# Invert purchase frequency, since lesser number of days between purchases indicates stronger engagement with the brand.

df["engagement_score"] = 1 - df["frequency_score"]

display(
    df[
        [
            "purchase_score",
            "rating_score",
            "engagement_score"
        ]
    ].head(10)
)

,purchase_score,rating_score,engagement_score
0,0.28,0.62,0.958904
1,0.04,0.62,0.958904
2,0.46,0.62,0.980822
3,0.98,0.70,0.980822
4,0.62,0.54,0.000000
5,0.28,0.58,0.980822
6,0.98,0.64,0.753425
7,0.38,0.64,0.980822
8,0.16,0.52,0.000000
9,0.08,0.96,0.753425


### Step 4.5: Constructing the Customer Health Index

In [84]:
df["customer_health_index"] = (

    chi_weights["previous_purchases"]
    * df["purchase_score"]

    +

    chi_weights["purchase_frequency_days"]
    * df["engagement_score"]

    +

    chi_weights["review_rating"]
    * df["rating_score"]

) * 100

In [85]:
display(
    df[
        [
            "purchase_score",
            "rating_score",
            "engagement_score",
            "customer_health_index"
        ]
    ].head(10)
)
#A higher CHI indicates a loyal customer, engaged with the brand, likely to continue purchasing, less risky to lose. A lower CHI indicates a retention risk.

,purchase_score,rating_score,engagement_score,customer_health_index
0,0.28,0.62,0.958904,63.022437
1,0.04,0.62,0.958904,58.925585
2,0.46,0.62,0.980822,66.536552
3,0.98,0.70,0.980822,80.436064
4,0.62,0.54,0.000000,44.488770
5,0.28,0.58,0.980822,60.952414
6,0.98,0.64,0.753425,72.088509
7,0.38,0.64,0.980822,66.426684
8,0.16,0.52,0.000000,35.380719
9,0.08,0.96,0.753425,76.817302


### Customer Health Index(CHI) Summary

The Customer Health Index combines purchase behaviour, engagement, and customer satisfaction and unlike arbitrary weighted scores, the contribution of each component is derived from its observed relationship with purchase amount.


## Feature 5: Promotion Dependency Index (PDI)

The Promotion Dependency Index (PDI) estimates the extent to which a **customer's purchasing behaviour** appears to rely on **promotional incentives**. Candidate variables are selected based on their relationship with **Discount Applied**. The variables are:
- Subscription Status
- Previous Purchases
- Purchase Frequency (Days)

### Step 5.1: Measuring Relationships with Discount Behaviour

In [86]:
# Convert binary variables into numerical format

df["discount_flag"] = df["discount_applied"].map({
    "Yes": 1,
    "No": 0
})

df["subscription_flag"] = df["subscription_status"].map({
    "Yes": 1,
    "No": 0
})

# Variables contributing to Promotion Dependency Index

pdi_variables = [
    "subscription_flag",
    "previous_purchases",
    "purchase_frequency_days"
]

# Calculate correlations(Pearsons) with Discount Applied

pdi_correlations = (
    df[pdi_variables + ["discount_flag"]]
      .corr()["discount_flag"]
      .drop("discount_flag")
)

display(
    pdi_correlations.to_frame(
        name="Correlation"
    )
)

,Correlation
subscription_flag,0.700202
previous_purchases,0.023537
purchase_frequency_days,-0.010300


In [87]:
# Preserve original correlations

original_pdi_correlations = pdi_correlations.copy()

# Convert to absolute values

absolute_pdi_correlations = pdi_correlations.abs()

display(
    absolute_pdi_correlations.to_frame(
        name="Absolute Correlation"
    )
)

,Absolute Correlation
subscription_flag,0.700202
previous_purchases,0.023537
purchase_frequency_days,0.010300


### Step 5.2: Converting Correlations into Weights

In [88]:
pdi_weights = (
    absolute_pdi_correlations /
    absolute_pdi_correlations.sum()
)

display(
    (pdi_weights * 100)
    .round(2)
    .to_frame(name="Weight (%)")
)

,Weight (%)
subscription_flag,95.39
previous_purchases,3.21
purchase_frequency_days,1.40


In [89]:
#Tabular summary of correlation and corresponding weights.
pdi_summary = pd.DataFrame({

    "Correlation":
        original_pdi_correlations.round(3),

    "Absolute Correlation":
        absolute_pdi_correlations.round(3),

    "Weight (%)":
        (pdi_weights * 100).round(2)

})

display(pdi_summary)

,Correlation,Absolute Correlation,Weight (%)
subscription_flag,0.700,0.700,95.39
previous_purchases,0.024,0.024,3.21
purchase_frequency_days,-0.010,0.010,1.40


### Step 5.3: Preparing the Variables

In [90]:
#Continuous variables are expressed as a proportion of their maximum observed value for better comparison with the subcription flag, which is binary.

# Subscription Status is already binary

df["subscription_score"] = df["subscription_flag"]

# Normalize Previous Purchases

df["previous_purchase_score"] = (
    df["previous_purchases"] /
    df["previous_purchases"].max()
)

# Normalize Purchase Frequency

df["promo_frequency_score_1"] = (
    df["purchase_frequency_days"] /
    df["purchase_frequency_days"].max()
)

### Step 5.4: Orienting the Variables

In [91]:
# Invert purchase frequency, since lesser number of days between purchases indicates greater promotional dependence.
df["promo_frequency_score"] = (
    1 -
    df["promo_frequency_score_1"]
)

display(
    df[
        [
            "subscription_score",
            "previous_purchase_score",
            "promo_frequency_score"
        ]
    ].head(10)
)

,subscription_score,previous_purchase_score,promo_frequency_score
0,1,0.28,0.958904
1,1,0.04,0.958904
2,1,0.46,0.980822
3,1,0.98,0.980822
4,1,0.62,0.000000
5,1,0.28,0.980822
6,1,0.98,0.753425
7,1,0.38,0.980822
8,1,0.16,0.000000
9,1,0.08,0.753425


### Step 5: Constructing the Promotion Dependency Index

In [92]:
df["promotion_dependency_index"] = (
    pdi_weights["subscription_flag"]* df["subscription_score"]
    +
    pdi_weights["previous_purchases"]* df["previous_purchase_score"]
    +
    pdi_weights["purchase_frequency_days"]* df["promo_frequency_score"]
)*100

In [93]:
display(
    df[
        [
            "subscription_score",
            "previous_purchase_score",
            "promo_frequency_score",
            "promotion_dependency_index"
        ]
    ].head(10)
)

,subscription_score,previous_purchase_score,promo_frequency_score,promotion_dependency_index
0,1,0.28,0.958904,97.633643
1,1,0.04,0.958904,96.864079
2,1,0.46,0.980822,98.241571
3,1,0.98,0.980822,99.908959
4,1,0.62,0.000000,97.378309
5,1,0.28,0.980822,97.664398
6,1,0.98,0.753425,99.589871
7,1,0.38,0.980822,97.985050
8,1,0.16,0.000000,95.903312
9,1,0.08,0.753425,96.704008


### Promotion Dependency Index Summary

The Promotion Dependency Index combines behavioural indicators associated with promotional engagement into a single interpretable business metric.

The resulting index will be used alongside the Customer Health Index to distinguish genuinely loyal customers from customers whose purchasing behaviour appears to be promotion-driven.

# Phase 3: Defining Customer Loyalty

One of the primary objectives of this project is to identify genuinely loyal customers. However, the dataset contains no explicit loyalty label or churn indicator. So, using the engineered features, I will create two loyalty definitions and compare them.

## Loyalty Definition 1: Commercial Loyalty

Commercial Loyalty defines loyal customers as those who maintain a strong relationship with the brand while exhibiting relatively low dependence on promotional incentives.

In [94]:
# CHI Quartiles(3 Quartiles for a better distribution since retention strategies will be tailored for moderate consumers)
chi_q1 = df["customer_health_index"].quantile(0.25)
chi_q3 = df["customer_health_index"].quantile(0.75)

df["chi_tier"] = np.where(
    df["customer_health_index"] < chi_q1,
    "Low",
    np.where(
        df["customer_health_index"] >= chi_q3,
        "High",
        "Moderate"
    )
)
#Promotion Dependency Tier(2 Tiers since it was observed that pdi lied in 2 categories)
df["promotion_tier"] = np.where(
    df["promotion_dependency_index"] < 5,
    "Low",
    "High"
)

In [95]:
# Evaluating Commercial Loyalty
df["commercial_loyalty"] = np.where(
    (df["chi_tier"] == "High")&(df["promotion_tier"] == "Low"),
    "High",
    np.where(
        (df["chi_tier"] == "Low")& (df["promotion_tier"] == "High"),
        "Low",
        "Moderate")
)
display(
    df[
        [
            "customer_health_index",
            "promotion_dependency_index",
            "chi_tier",
            "promotion_tier",
            "commercial_loyalty"
        ]
    ].head(10)
)
print(df["commercial_loyalty"].value_counts())

,customer_health_index,promotion_dependency_index,chi_tier,promotion_tier,commercial_loyalty
0,63.022437,97.633643,Moderate,High,Moderate
1,58.925585,96.864079,Low,High,Low
2,66.536552,98.241571,Moderate,High,Moderate
3,80.436064,99.908959,High,High,Moderate
4,44.488770,97.378309,Low,High,Low
5,60.952414,97.664398,Low,High,Low
6,72.088509,99.589871,Moderate,High,Moderate
7,66.426684,97.985050,Moderate,High,Moderate
8,35.380719,95.903312,Low,High,Low
9,76.817302,96.704008,Moderate,High,Moderate


commercial_loyalty
Moderate    2950
High         704
Low          246
Name: count, dtype: int64


In [96]:
# Commercial Loyalty Distribution
commercial_distribution = (df["commercial_loyalty"]
    .value_counts()
    .rename_axis("Commercial Loyalty")
    .reset_index(name="Customer Count")
)
commercial_distribution["Percentage"] = (
    commercial_distribution["Customer Count"]/commercial_distribution["Customer Count"].sum()*100).round(2)

display(commercial_distribution)
print()

,Commercial Loyalty,Customer Count,Percentage
0,Moderate,2950,75.64
1,High,704,18.05
2,Low,246,6.31


## Loyalty Definition 2: Behavioural Loyalty

Behavioural Loyalty focuses purely on customer engagement and satisfaction.

In [97]:
#Dataset medians(Previous Purchases and Purchase Frequency)
purchase_median = df["previous_purchases"].median()

frequency_median = df["purchase_frequency_days"].median()

print(f"Previous Purchases Median: {purchase_median}")
print(f"Purchase Frequency Median: {frequency_median}")

Previous Purchases Median: 25.0
Purchase Frequency Median: 30.0


In [98]:
# Behavioural Loyalty (Three-Tier Segmentation)
behavioural_score = (
    (df["previous_purchases"] >= purchase_median).astype(int)
    +
    (df["purchase_frequency_days"] <= frequency_median).astype(int)
    +
    (df["satisfaction_flag"] == "Satisfied").astype(int)
)

df["behavioural_loyalty"] = np.where(
    behavioural_score == 3,
    "High",
    np.where(
        behavioural_score == 2,
        "Moderate",
        "Low"
    )
)

display(
    df[
        [
            "previous_purchases",
            "purchase_frequency_days",
            "satisfaction_flag",
            "behavioural_loyalty"
        ]
    ].head(10)
)

,previous_purchases,purchase_frequency_days,satisfaction_flag,behavioural_loyalty
0,14,15,Needs Attention,Low
1,2,15,Needs Attention,Low
2,23,7,Needs Attention,Low
3,49,7,Needs Attention,Moderate
4,31,365,Needs Attention,Low
5,14,7,Needs Attention,Low
6,49,90,Needs Attention,Low
7,19,7,Needs Attention,Low
8,8,365,Needs Attention,Low
9,4,90,Satisfied,Low


In [99]:
# Behavioural Loyalty Distribution
behavioural_distribution = (
    df["behavioural_loyalty"]
      .value_counts()
      .rename_axis("Behavioural Loyalty")
      .reset_index(name="Customer Count")
)

behavioural_distribution["Percentage"] = (behavioural_distribution["Customer Count"]/ behavioural_distribution["Customer Count"].sum()* 100).round(2)
display(behavioural_distribution)

,Behavioural Loyalty,Customer Count,Percentage
0,Low,1822,46.72
1,Moderate,1498,38.41
2,High,580,14.87


## Evaluating the Two Loyalty Definitions

Since loyalty is not directly available within the dataset, both definitions are evaluated using objective business criteria.

Three evaluation measures are used:

1. Correlation with Purchase Amount.
2. Revenue separation between Loyal and Not Loyal customers.
3. Internal consistency of the variables used within each definition.

The definition demonstrating stronger explanatory power will be selected for the remainder of the project.

In [100]:
commercial_summary = (

    df

    .groupby("commercial_loyalty")["purchase_amount_usd"]

    .mean()

    .round(2)

)

behavioural_summary = (

    df

    .groupby("behavioural_loyalty")["purchase_amount_usd"]

    .mean()

    .round(2)

)

print("Commercial Loyalty")

display(commercial_summary)

print("\nBehavioural Loyalty")

display(behavioural_summary)

Commercial Loyalty


commercial_loyalty
High        60.20
Low         58.92
Moderate    59.73
Name: purchase_amount_usd, dtype: float64


Behavioural Loyalty


behavioural_loyalty
High        60.69
Low         59.51
Moderate    59.71
Name: purchase_amount_usd, dtype: float64

In [101]:
# Calculate Revenue Gap
commercial_gap = (commercial_summary["High"]-commercial_summary["Low"])
behavioural_gap = (behavioural_summary["High"]-behavioural_summary["Low"])

# Calculate average CHI and PDI for Highly Loyal customers only
commercial_chi = (df[df["commercial_loyalty"] == "High"]["customer_health_index"].mean())

commercial_pdi = (df[df["commercial_loyalty"] == "High"]["promotion_dependency_index"].mean())

behavioural_chi = (df[df["behavioural_loyalty"] == "High"]["customer_health_index"].mean())

behavioural_pdi = (df[df["behavioural_loyalty"] == "High"]["promotion_dependency_index"].mean())

# Comparison Table

comparison = pd.DataFrame({
    "Definition": ["Commercial Loyalty","Behavioural Loyalty"],
    
    "Revenue Gap": [round(commercial_gap, 2),round(behavioural_gap, 2)],

    "Average CHI": [round(commercial_chi, 2),round(behavioural_chi, 2)],

    "Average PDI": [round(commercial_pdi, 2),round(behavioural_pdi, 2)]
})

display(comparison)

#The table displays parameters for highly loyal consumers

,Definition,Revenue Gap,Average CHI,Average PDI
0,Commercial Loyalty,1.28,86.16,3.42
1,Behavioural Loyalty,1.18,86.82,30.72


### Selecting the Final Loyalty Definition

The two competing loyalty definitions are compared using three business criteria:

1. **Revenue Gap** — Measures how effectively the definition separates high-value customers from lower-value customers.

2. **Average Customer Health Index (CHI)** — Indicates the overall strength of the customer relationship among customers classified as loyal.

3. **Average Promotion Dependency Index (PDI)** — Indicates the extent to which loyal customers rely on promotional incentives.
The preferred loyalty definition should:

- Produce a larger Revenue Gap,
- Identify customers with a higher Customer Health Index,
- Identify customers with a lower Promotion Dependency Index.


In [102]:
# Based on the comparison above, Commercial Loyalty
# demonstrates stronger commercial value while identifying
# customers with higher CHI and lower PDI.

selected_definition = "Commercial Loyalty"

print(f"Selected Loyalty Definition: {selected_definition}")

Selected Loyalty Definition: Commercial Loyalty


In [103]:
#Exporting the Final Dataset
output_file = "customer_value_analytics_cleaned.csv"

df.to_csv(output_file, index=False)

print(f"Dataset exported successfully as '{output_file}'.")

Dataset exported successfully as 'customer_value_analytics_cleaned.csv'.
